# NL-to-SQL pipeline walkthrough

Tracing one question through the real `nl2sql/` functions with fake data.

Schema → prompt → extract SQL → postprocess → validate → repair → score.

No model or database connection needed — model output is hardcoded.

In [ ]:
import os, sys, shutil
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_dir = Path("/content/NLtoSQL")
    if not (repo_dir / "nl2sql").exists():
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        os.system(f"git clone https://github.com/MacKenzieOBrian/NLtoSQL.git {repo_dir}")
    os.chdir(repo_dir)
    os.system("pip -q install -r requirements.txt")

sys.path.insert(0, str(Path.cwd()))
print("working directory:", Path.cwd())
print("in Colab:", IN_COLAB)

In [ ]:
# real imports — no torch, no DB connection needed for these
from nl2sql.core.prompting   import make_few_shot_messages
from nl2sql.core.postprocess import normalize_sql, first_select_only, guarded_postprocess, RANKING_HINT_RE
from nl2sql.core.validation  import parse_schema_text, validate_sql, validate_constraints
from nl2sql.core.llm         import extract_first_select
from nl2sql.agent.agent_tools import schema_to_text

print("imports ok")

---
## 1. Schema text

`schema_to_text()` (`agent_tools.py`) converts the schema dict returned by `build_schema_summary()` into the plain text string that gets prepended to every prompt.

In [ ]:
# fake schema dict — same structure as what build_schema_summary() returns
# (nl2sql/core/schema.py queries INFORMATION_SCHEMA at runtime)
schema_cache = {
    "tables": [
        {"name": "customers",    "columns": [{"name": "customerNumber"}, {"name": "customerName"}, {"name": "country"}, {"name": "creditLimit"}]},
        {"name": "orders",       "columns": [{"name": "orderNumber"}, {"name": "customerNumber"}, {"name": "status"}, {"name": "orderDate"}]},
        {"name": "orderdetails", "columns": [{"name": "orderNumber"}, {"name": "productCode"}, {"name": "quantityOrdered"}, {"name": "priceEach"}]},
        {"name": "products",     "columns": [{"name": "productCode"}, {"name": "productName"}, {"name": "productLine"}, {"name": "MSRP"}]},
        {"name": "payments",     "columns": [{"name": "customerNumber"}, {"name": "checkNumber"}, {"name": "amount"}, {"name": "paymentDate"}]},
    ],
    "foreign_keys": [
        {"table": "orders",    "column": "customerNumber", "ref_table": "customers", "ref_column": "customerNumber"},
        {"table": "payments",  "column": "customerNumber", "ref_table": "customers", "ref_column": "customerNumber"},
        {"table": "orderdetails", "column": "orderNumber", "ref_table": "orders",    "ref_column": "orderNumber"},
    ],
}

schema_text = schema_to_text(schema_cache)
print("schema_to_text() ->")
print()
print(schema_text)

---
## 2. Building the prompt

`make_few_shot_messages()` (`prompting.py`) builds the message list the model sees.  
Each exemplar becomes a user/assistant pair sitting between the schema and the question.

In [ ]:
NLQ = "Top 3 countries by number of orders."

EXEMPLARS = [
    {"nlq": "List all customer names in Spain.",
     "sql": "SELECT customerName FROM customers WHERE country = 'Spain';"},
    {"nlq": "How many products are in each product line?",
     "sql": "SELECT productLine, COUNT(*) FROM products GROUP BY productLine;"},
    {"nlq": "What is the total payment amount received?",
     "sql": "SELECT SUM(amount) FROM payments;"},
]

msgs_k0 = make_few_shot_messages(schema=schema_text, exemplars=[],        nlq=NLQ)
msgs_k3 = make_few_shot_messages(schema=schema_text, exemplars=EXEMPLARS, nlq=NLQ)

def show_messages(msgs, label):
    print(f"--- {label} ({len(msgs)} messages) ---")
    for m in msgs:
        body = m['content'][:100].replace('\n', ' | ')
        print(f"  [{m['role'].upper():<10}]  {body}")
    print()

show_messages(msgs_k0, "k=0")
show_messages(msgs_k3, "k=3")

print(f"k=0: {len(msgs_k0)} messages  /  k=3: {len(msgs_k3)} messages")
print(f"each exemplar adds 2 messages (user + assistant), so k=3 adds 6")

---
## 3. Extracting SQL from raw model output

Models often return explanation text around the SQL.  
`extract_first_select()` (`llm.py`) strips markdown fences, then scans for the first `SELECT` that looks like a real table-backed query — it rejects candidates where `FROM` is followed by a stopword like "the" or "a".

In [ ]:
# typical raw output from a small model — explanation prefix + trailing prose
RAW_NOISY = """Sure! Here is the SQL for your question:

SELECT c.country, COUNT(o.orderNumber) AS order_count
FROM customers c
JOIN orders o ON c.customerNumber = o.customerNumber
GROUP BY c.country ORDER BY order_count DESC LIMIT 3;

This joins the customers and orders tables and returns the top 3 countries."""

print("raw model output:")
for line in RAW_NOISY.strip().split('\n'):
    print(f"  {line}")
print()

extracted = extract_first_select(RAW_NOISY)
print(f"extract_first_select() -> {extracted}")
print()

# prose-only text — no valid SELECT — returns None
RAW_NO_SQL = "I don't know how to answer that question."
print(f"no-SQL text     -> {extract_first_select(RAW_NO_SQL)!r}  (None = fallback to raw text)")

# markdown fences are stripped before scanning
RAW_FENCED = "```sql\nSELECT productName FROM products;\n```"
print(f"fenced SQL      -> {extract_first_select(RAW_FENCED)}")

# prose-FROM is rejected ("from the results" is not a table)
RAW_PROSE = "I can answer from the results I have."
print(f"prose FROM      -> {extract_first_select(RAW_PROSE)!r}  (rejected — 'the' is a stopword)")

---
## 4. Postprocessing

`guarded_postprocess()` (`postprocess.py`) applies two conservative transforms in order:

- `first_select_only` — re-extracts the first SELECT statement, stripping any trailing prose
- `_strip_order_by_limit` — removes ORDER BY / LIMIT unless `RANKING_HINT_RE` fires on the NLQ (keywords: top, highest, lowest, most, order, rank, …)

Both steps are OFF by default in primary evaluation runs (`apply_postprocess=False`). They are part of the optional reliability layer tested only as an extension.

In [ ]:
# NLQ with no ranking keyword → strip_order_by_limit should fire (ORDER BY removed)
NLQ_PP = "What is the total amount paid by each customer?"
SQL_PP  = "SELECT customerNumber, SUM(amount) AS total, paymentDate FROM payments GROUP BY customerNumber ORDER BY total DESC;"

print("=== no ranking keyword in NLQ ===")
print(f"input:  {SQL_PP}")
print(f"NLQ:    {NLQ_PP}")
print()

step1 = first_select_only(SQL_PP)
print(f"step 1 — first_select_only:  {'CHANGED' if step1 != SQL_PP else 'no change'}")

ranking_match = RANKING_HINT_RE.search(NLQ_PP)
print(f"step 2 — RANKING_HINT_RE:    {'fires — keep ORDER BY' if ranking_match else 'no match — strip ORDER BY / LIMIT'}")

result = guarded_postprocess(SQL_PP, NLQ_PP)
print(f"\nguarded_postprocess() -> {result}")
print()

# contrast: NLQ that does mention a ranking cue → ORDER BY preserved
NLQ_RANK = "What are the top 3 countries by total payments?"
result_rank = guarded_postprocess(SQL_PP, NLQ_RANK)
ranking_match2 = RANKING_HINT_RE.search(NLQ_RANK)
print(f"=== ranking keyword present ===")
print(f"NLQ:    {NLQ_RANK}")
print(f"RANKING_HINT_RE fires: {bool(ranking_match2)}  (keyword: '{ranking_match2.group(0)}')")
print(f"guarded_postprocess() -> {result_rank}")
print("ORDER BY and LIMIT preserved — ranking NLQ detected")

---
## 5. Schema validation

`validate_sql()` (`validation.py`) first parses the schema text into a lookup index via `parse_schema_text()`, then checks every table and column reference in the SQL.

In [ ]:
# first see what parse_schema_text does with the schema string
tables, table_cols = parse_schema_text(schema_text)
print("parse_schema_text() ->")
print(f"  tables: {sorted(tables)}")
print()
for tbl, cols in sorted(table_cols.items()):
    print(f"  {tbl}: {sorted(cols)}")
print()

# test three SQL statements against the schema index
test_cases = [
    ("good",
     "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY COUNT(*) DESC LIMIT 3;"),
    ("bad — 'country' not in orders",
     "SELECT country, COUNT(*) FROM orders GROUP BY country ORDER BY COUNT(*) DESC LIMIT 3;"),
    ("bad — unknown table 'invoices'",
     "SELECT customerNumber FROM invoices WHERE amount > 1000;"),
]

print("validate_sql() results:")
for label, sql in test_cases:
    r = validate_sql(sql, schema_text, nlq=NLQ)
    status = "PASS" if r['valid'] else f"FAIL  [{r['reason']}]"
    print(f"\n  {label}")
    print(f"  {status}")
    print(f"  sql: {sql}")

---
## 6. Constraint validation

`validate_constraints()` checks SQL shape against hints built from the NLQ.  
In `react_pipeline.py` these are derived by `_infer_constraints()` before each validate call — not hardcoded.

In [ ]:
# constraints that react_pipeline.py would infer for "Top 3 countries by number of orders"
constraints = {
    "agg":            "COUNT",
    "needs_group_by": True,
    "needs_order_by": True,
    "limit":          3,
}

sql_pass = "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY COUNT(*) DESC LIMIT 3;"
sql_fail = "SELECT country, COUNT(*) FROM customers GROUP BY country;"  # no ORDER BY, no LIMIT

print("constraints:", constraints)
print()
for label, sql in [("passing", sql_pass), ("failing (no ORDER BY, no LIMIT)", sql_fail)]:
    r = validate_constraints(sql, constraints)
    status = "PASS" if r['valid'] else f"FAIL  [{r['reason']}]"
    print(f"  {label}")
    print(f"  {status}")
    print(f"  {sql}")
    print()

In [ ]:
# required_output_fields constraint — check the SELECT list has the right columns
constraints_2 = {
    "required_output_fields": ["customerName"],
}

sql_good = "SELECT customerName, COUNT(*) FROM orders o JOIN customers c ON o.customerNumber = c.customerNumber GROUP BY customerName;"
sql_bad  = "SELECT customerNumber, COUNT(*) FROM orders GROUP BY customerNumber;"

print("constraints:", constraints_2)
print()
for label, sql in [("SELECT has customerName", sql_good), ("SELECT only has customerNumber", sql_bad)]:
    r = validate_constraints(sql, constraints_2)
    status = "PASS" if r['valid'] else f"FAIL  [{r['reason']}]"
    print(f"  {label}")
    print(f"  {status}")
    if not r['valid'] and 'missing_fields' in r:
        print(f"  missing_fields: {r['missing_fields']}")
    print(f"  {sql}")
    print()

---
## 7. Repair scenario

When validation fails, `react_pipeline.py` appends a hint to the repair prompt and calls the model again.  
This cell simulates one repair cycle — the "model response" is hardcoded.

In [ ]:
# attempt 1 — first SQL the model produces
attempt_1 = "SELECT country, COUNT(*) FROM orders GROUP BY country ORDER BY COUNT(*) DESC LIMIT 3;"
print("attempt 1:")
print(f"  {attempt_1}")
r1 = validate_sql(attempt_1, schema_text, nlq=NLQ)
print(f"  validate_sql -> {r1}")
print()

# react_pipeline.py builds a hint string from the failure reason and adds it to the repair prompt
hint = f"[{r1['reason']}] -- schema shows 'country' is in 'customers', not in 'orders'"
print("hint sent to model in repair prompt:")
print(f"  {hint}")
print()

# attempt 2 — "model" produces corrected SQL (hardcoded here)
attempt_2 = "SELECT c.country, COUNT(*) AS n FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY n DESC LIMIT 3;"
print("attempt 2 (repaired):")
print(f"  {attempt_2}")
r2 = validate_sql(attempt_2, schema_text, nlq=NLQ)
print(f"  validate_sql -> {r2}")
print()
print("repairs used: 1 of 1  (max_repairs=1 in ReactAblationConfig default)")

In [ ]:
# EM scoring — normalize_sql() strips semicolons, lowercases, collapses whitespace
gold_sql = "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY COUNT(*) DESC LIMIT 3;"

print("normalize_sql() output:")
print(f"  gold:      {normalize_sql(gold_sql)}")
print(f"  attempt_1: {normalize_sql(attempt_1)}")
print(f"  attempt_2: {normalize_sql(attempt_2)}")
print()

print("EM / EX per attempt:")
for label, pred, ex in [
    ("attempt_1 (schema error)", attempt_1, False),
    ("attempt_2 (repaired)",     attempt_2, True),
]:
    em = normalize_sql(pred) == normalize_sql(gold_sql)
    print(f"  {label}:  EM={em}  EX={ex}  VA=True")
print()
print("attempt_2 has EX=True (returns same rows) but EM=False (different alias 'n' vs no alias)")
print("-> EX is the primary metric; EM is too strict for equivalent SQL")

---
## 8. Results

Means across 3 seeds × 200 items = 600 scored pairs per condition.  
All 8 base/QLoRA conditions now have full seed coverage (7, 17, 27).

In [ ]:
# all 8 conditions — means from results/analysis/stats_mean_median_shapiro.csv
conditions = [
    ("Llama Base  k=0", 0.765, 0.005, 0.410, None),
    ("Llama Base  k=3", 0.870, 0.298, 0.517, 0.559),
    ("Llama QLoRA k=0", 0.815, 0.000, 0.440, None),
    ("Llama QLoRA k=3", 0.855, 0.285, 0.465, None),
    ("Qwen  Base  k=0", 0.805, 0.010, 0.480, None),
    ("Qwen  Base  k=3", 0.892, 0.357, 0.610, 0.667),
    ("Qwen  QLoRA k=0", 0.805, 0.010, 0.435, None),
    ("Qwen  QLoRA k=3", 0.895, 0.302, 0.530, 0.566),
]

print(f"  {'Condition':<20}  {'VA':>6}  {'EM':>6}  {'EX':>6}  {'TS':>6}")
print(f"  {'-'*20}  {'------':>6}  {'------':>6}  {'------':>6}  {'------':>6}")
for label, va, em, ex, ts in conditions:
    ts_str = f"{ts:.3f}" if ts else "  N/A"
    flag   = "  <--" if "k=3" in label else ""
    print(f"  {label:<20}  {va:>6.3f}  {em:>6.3f}  {ex:>6.3f}  {ts_str:>6}{flag}")

---
## 9. Statistical tests

Paired t-tests on per-example 0/1 scores, matched by (seed, example_id).  
Shapiro-Wilk rejects normality everywhere (expected for binary data) — CLT justifies t-test at n=400–600.

In [ ]:
# paired t-test summary — from results/analysis/stats_paired_ttests.csv
tests = [
    ("Llama Base k0->k3",     "EX", 0.410, 0.517, 1.10e-11, True),
    ("Llama QLoRA k0->k3",    "EX", 0.440, 0.465, 0.0547,   False),   # NOT significant
    ("Qwen Base k0->k3",      "EX", 0.490, 0.610, 8.21e-09, True),
    ("Qwen QLoRA k0->k3",     "EX", 0.435, 0.530, 1.37e-09, True),
    ("Llama Base vs QLoRA k3","EX", 0.517, 0.465, 7.40e-04, True),    # core claim
    ("Qwen Base vs QLoRA k3", "EX", 0.610, 0.530, 8.60e-06, True),    # core claim
]

print(f"  {'comparison':<26}  {'M':<3}  {'left':>6}  {'right':>6}  {'delta':>7}  {'p':>10}  sig?")
print(f"  {'-'*26}  ---  {'------':>6}  {'------':>6}  {'-------':>7}  {'----------':>10}  ----")
for comp, m, lv, rv, p, sig in tests:
    delta = (rv - lv) * 100
    p_str = f"{p:.2e}" if p < 0.001 else f"{p:.4f}"
    note  = "yes" if sig else "NO (p=0.055)"
    print(f"  {comp:<26}  {m:<3}  {lv:>6.3f}  {rv:>6.3f}  {delta:>+6.1f}pp  {p_str:>10}  {note}")

print()
print("Llama QLoRA k0->k3 EX gain is NOT significant — fine-tuning may suppress ICL benefit")
print()

# ReAct (3 seeds: 7, 17, 27)
print("ReAct mean vs Llama Base k=3  (3 seeds, n=600 matched items):")
for m, bv, rv in zip(["VA","EM","EX","TS"], [0.870,0.298,0.517,0.559], [0.925,0.458,0.628,0.661]):
    print(f"  {m}: {bv:.3f} -> {rv:.3f}  ({(rv-bv)*100:+.1f}pp)")
print("  [ReAct means computed from seeds 7, 17, 27]")

---
## 10. Summary

Main result: **k=3 prompting beats QLoRA at k=3** on both models, statistically significant.

The k=0→k=3 jump is the clearest story — EM goes from near-zero to ~30% just by adding 3 examples. QLoRA doesn't add to that; the Llama fine-tuned model's k=0→k=3 EX gain is borderline (p=0.055), suggesting fine-tuning partially suppresses in-context learning.

ReAct (Llama + QLoRA, seeds 7/17/27) reaches **EX=0.628** and **TS=0.661**. Relative to Llama Base k=3 (EX=0.517), this is a **+11.1pp EX** gain on a 3-seed evaluation.